In a large-scale data processing project, multiple PySpark notebooks require the same set of custom transformation functions, such as date formatting, null handling, and data validation. Instead of duplicating the code across notebooks, a Python class is created to store these reusable functions. This ensures consistency, reduces maintenance effort, and improves code readability across the project.

## **Python Class**

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
class DataValidation:
    def __init__(self,df):
        self.df= df

    def dedup(self, keyCol, cdcCol):
        df= self.df.withColumn("dedup", row_number().over(Window.partitionBy(keyCol).orderBy(desc(cdcCol))))
        df= df.filter(col('dedup')==1).drop('dedup')

        return df
    
    def removeNulls(self,nullCol):
        df= self.df.filter(col(nullCol).isNotNull())

        return df

In [0]:
df = spark.createDataFrame([(1, 100, '2020-01-01'), (2, 200, '2020-01-02'), (3, 300, '2020-01-03'), (4, 400, '2020-01-04'), (5, 500, '2020-01-05'),(2, 500, '2020-01-06')],["order_id", "amount", "order_date"])

display(df)

In [0]:
cls_obj = DataValidation(df)

In [0]:

df_dedup = cls_obj.dedup('order_id','order_date')
display(df_dedup)
